In [ ]:
%cd ..

# Principal Component Analysis

In [ ]:
from typing import Optional, Union
from dataclasses import dataclass
import pickle
import numba as nb
import numpy as np
import scipy.stats as stats
import mdtraj as md
from sklearn.decomposition import PCA

@dataclass(slots=True, frozen=True)
class NBConfig:
    # PCA parameters
    n_components: Optional[int] = None # If None, all components are kept
    evr_threshold: Optional[float] = None # If set, select components to cover this EVR
    whiten: bool = False

    @property
    def n_components_effective(self) -> Optional[Union[int, float]]:
        """Return the effective number of components to keep based on the configuration."""
        if self.evr_threshold is not None:
            return self.evr_threshold
        return self.n_components

In [ ]:
def _compute_pca(data: np.ndarray, config: NBConfig):
    """Compute PCA on the given data."""
    flat_data = data.reshape(data.shape[0], -1)

    pca = PCA(n_components=config.n_components_effective, whiten=config.whiten)
    pca.fit(flat_data)

    components = pca.transform(flat_data)

    n_components = components.shape[1]
    explained_variance = pca.explained_variance_[:n_components]
    explained_variance_ratio = pca.explained_variance_ratio_[:n_components]
    cumulative_explained_variance_ratio = np.cumsum(explained_variance_ratio)

    # Compute the loadings and the contribution
    loadings = (pca.components_.T * np.sqrt(pca.explained_variance_)).T
    loadings_sq = loadings ** 2
    contributions = loadings_sq / loadings_sq.sum(axis=-1, keepdims=True)

    loadings = loadings[:n_components]
    contributions = contributions[:n_components]

    loadings = loadings.reshape(n_components, -1, data.shape[-1])
    contributions = contributions.reshape(n_components, -1, data.shape[-1])


    stats = {
        "explained_variance": explained_variance,
        "explained_variance_ratio": explained_variance_ratio,
        "cumulative_explained_variance_ratio": cumulative_explained_variance_ratio,
        "loadings": loadings,
        "contributions": contributions,
        "n_components": n_components
    }
    return components, stats

def compute_pca(features: dict[str, np.ndarray],config: NBConfig):
    """Compute PCA"""
    results_data = {}
    results_stats = {}
    for representation, data in features.items():
        components, stats = _compute_pca(data, config)
        results_data[representation] = components
        results_stats[representation] = stats
    return results_data, results_stats

## 2 Components

In [ ]:
top_pdb = "examples/_structure.pdb"
traj_dcd = "examples/_trajectory.dcd"

config = NBConfig(n_components=2, whiten=False)

In [ ]:
with open("examples/features.pkl", "rb") as h:
    features = pickle.load(h)["data"]

components, stats = compute_pca(features, config)

with open("examples/pcs_2.pkl", "wb") as h:
    pickle.dump({
        "data": components,
        "stats": stats
    }, h, protocol=pickle.HIGHEST_PROTOCOL)

## 0.95 Explained Variance Ratio & Whiten (TICA Preprocessing)

In [ ]:
top_pdb = "/scratch/agiottonini/DEShaw_simulations/nsp13/open/DESRES-Trajectory_sarscov2-12212701-5-1-no-water/sarscov2-12212701-5-1-no-water/DESRES-Trajectory_sarscov2-12212701-5-1-no-water.pdb"
traj_dcd = "/scratch/agiottonini/DEShaw_simulations/nsp13/open/DESRES-Trajectory_sarscov2-12212688-5-2-no-water/sarscov2-12212688-5-2-no-water/sarscov2-12212688-5-2-no-water-0000.dcd"

config = NBConfig(evr_threshold=0.95, whiten=True)

In [ ]:
with open("examples/features.pkl", "rb") as h:
    features = pickle.load(h)["data"]

components, stats = compute_pca(features, config)

with open("examples/pcs_preprocessing.pkl", "wb") as h:
    pickle.dump({
        "data": components,
        "stats": stats
    }, h, protocol=pickle.HIGHEST_PROTOCOL)